# Task 1 - Learn the Basics of PyTorch

## Multi Layer Perceptron

In [71]:
import torch
from torch import nn
from torchvision.datasets import FashionMNIST, CIFAR10
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset, random_split

In [72]:
print("---------------------------------------------------------------------")
print(f"PYTORCH Version: {torch.__version__}")
# setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DEVICE: {device}')

#Additional Info when using cuda
if device.type == 'cuda':
  print(f"GPU NAME: {torch.cuda.get_device_name(0)}")
  print('Memory Usage:')
  print('  - Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
  print('  - Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')
  print(f"---------------------------------------------------------------------\n")

---------------------------------------------------------------------
PYTORCH Version: 2.1.2
DEVICE: cuda
GPU NAME: NVIDIA GeForce RTX 4060
Memory Usage:
  - Allocated: 0.7 GB
  - Cached:    1.1 GB
---------------------------------------------------------------------



### Global Variables

In [73]:
NUM_CLASSES   = 10      # 10
NUM_FEATURES  = 28*28   # 28x28 image
RANDOM_SEED   = 42
BATCH_SIZE    = 64

In [74]:
# Loading Data - Taken from https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
train_data = FashionMNIST(
  root="data",
  train=True,
  download=True,
  transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = FashionMNIST(
  root="data",
  train=False,
  download=True,
  transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_data, val_data = random_split(
  train_data, 
  [0.9, 0.1], 
  generator=torch.Generator().manual_seed(RANDOM_SEED)
)

In [75]:
def move_to_gpu(data, device, shuffle=False):
  X, y = next(iter(DataLoader(data,batch_size=len(data))))
  return DataLoader(
    TensorDataset(X.to(device), y.to(device)),
    batch_size=BATCH_SIZE,
    shuffle=shuffle
  )

In [76]:

train_dataloader  = move_to_gpu(train_data, device, shuffle=True)
test_dataloader   = move_to_gpu(test_data, device, shuffle=False)
val_dataloader    = move_to_gpu(val_data, device, shuffle=False)

### Helper Functions

In [77]:
def move_to_gpu(data, device, shuffle=False):
  X, y = next(iter(DataLoader(data,batch_size=len(data))))
  return DataLoader(
    TensorDataset(X.to(device), y.to(device)),
    batch_size=BATCH_SIZE,
    shuffle=shuffle
  )

In [78]:
# Should be Train: 54000 Validation: 6000 Test: 10000
print(f"Train: {len(train_data)}  Validation: {len(val_data)}  Test: {len(test_data)}") 

Train: 54000  Validation: 6000  Test: 10000


In [79]:
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Model

### Architecture

In [80]:
class FashionModel(nn.Module):
  def __init__(self, input_features, output_features, hidden_units=8, activation_fn=nn.ReLU,
               num_hidden_layers=2, dropout=0.0):
    super().__init__()
    layers = [nn.Flatten()]
    in_features = input_features

    for _ in range(num_hidden_layers):
      layers.append(nn.Linear(in_features, hidden_units))
      layers.append(activation_fn())
      if dropout > 0.0:
        layers.append(nn.Dropout(dropout))
      in_features = hidden_units
      
    layers.append(nn.Linear(in_features, output_features))
    self.layer_stack = nn.Sequential(*layers)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.layer_stack(x)

### Initialisation

In [81]:
def apply_init(model: nn.Module, method: str):
  if method == "default":
    return
  
  for layer in model.layer_stack:
    if isinstance(layer, nn.Linear):
      if method == "kaiming_normal":
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
      elif method == "normal_0.01":
        nn.init.normal_(layer.weight, mean=0.0, std=0.01)
      elif method == "zeros":
        nn.init.zeros_(layer.weight)
      nn.init.zeros_(layer.bias)

### Loss

In [82]:
loss_fn = nn.CrossEntropyLoss()

## Training and Validation Function

In [83]:
def train_and_validate(model: nn.Module, optimizer, epochs=100) -> dict[str,list]:
  history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
  print(f"{'Epoch':<8}{'Train Loss':<12}{'Train Acc':<12}{'Val Loss':<12}{'Val Acc'}")

  for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    train_correct = 0
    for X, y in train_dataloader:
      y_pred = model(X)
      loss = loss_fn(y_pred, y)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      epoch_loss += loss.item()
      train_correct += (y_pred.argmax(dim=1) == y).sum().item()

    val_loss = 0.0
    val_correct = 0
    model.eval()
    with torch.inference_mode():
      for X, y in val_dataloader:
        val_pred = model(X)
        val_loss += loss_fn(val_pred, y).item()
        val_correct += (val_pred.argmax(dim=1) == y).sum().item()

    train_loss = epoch_loss / len(train_dataloader)
    train_acc = 100 * train_correct / len(train_data)
    val_loss = val_loss / len(val_dataloader)
    val_acc = 100 * val_correct / len(val_data)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"{epoch:<8}{train_loss:<12.5f}{train_acc:<12.2f}{val_loss:<12.5f}{val_acc:.2f}")
  return history

## Testing
This is only run once after the configurations have been set.
In short, the top 3 performers on the validation set are tested

In [84]:
def test(model: nn.Module):
  test_loss = 0.0
  test_correct = 0
  model.eval()
  with torch.inference_mode():
    for X, y in test_dataloader:
      test_pred = model(X)
      test_loss += loss_fn(test_pred, y).item()
      test_correct += (test_pred.argmax(dim=1) == y).sum().item()

  # print(f"\nTest loss: {test_loss / len(test_dataloader):.5f}  "
  #       f"Test acc: {100 * test_correct / len(test_data):.2f}%")

  return test_loss / len(test_dataloader), 100 * test_correct / len(test_data)

## Run Script

In [85]:
# configs = [
#   # {"name": "default",   "lr": 0.1,  "hidden_units": 16, "activation_fn": nn.ReLU, "optimizer": torch.optim.SGD},
#   # {"name": "lr_0.01",   "lr": 0.01, "hidden_units": 16, "activation_fn": nn.ReLU, "optimizer": torch.optim.SGD},      
#   # {"name": "hidden_64", "lr": 0.1,  "hidden_units": 64, "activation_fn": nn.ReLU, "optimizer": torch.optim.SGD}     Note: this is old experimentation, I found that adding more hidden units very slightly improves test accuracy
# ]

activation_functions = {
  "relu":         nn.ReLU,
  "leaky_relu":   nn.LeakyReLU,
  "gelu":         nn.GELU
}

optimizers_and_lrs = [
  {"name": "sgd_momentum", "fn": torch.optim.SGD,   "kwargs": {"lr": 0.01, "momentum": 0.9}},
  {"name": "adam",         "fn": torch.optim.Adam,  "kwargs": {"lr": 0.001}},
  {"name": "adamw",        "fn": torch.optim.AdamW, "kwargs": {"lr": 0.001}},
]

BASE = {
  "activation_fn":     nn.GELU,
  "optimizer":         torch.optim.AdamW,
  "optimizer_kwargs":  {"lr": 1e-3, "weight_decay": 0.01},
  "hidden_units":      64,
  "num_hidden_layers": 2,
  "dropout":           0.0,
  "init":              "default",
}

configs = []

for activation_name, activation_fn in activation_functions.items():
  for opt in optimizers_and_lrs:
    configs.append({
      **BASE,
      "name":             f"{activation_name}_{opt['name']}",
      "activation_fn":    activation_fn,
      "optimizer":        opt["fn"],
      "optimizer_kwargs": opt["kwargs"],
    })

configs += [
  {**BASE, "name": "lr3e-3",         "optimizer_kwargs": {"lr": 3e-3, "weight_decay": 0.01}},
  {**BASE, "name": "wd0.1",          "optimizer_kwargs": {"lr": 1e-3, "weight_decay": 0.1}},
  {**BASE, "name": "dropout0.3",     "dropout": 0.3},
  {**BASE, "name": "init_normal",    "init": "normal_0.01"},
  {**BASE, "name": "init_kaiming",   "init": "kaiming_normal"},
  {**BASE, "name": "hidden256",       "hidden_units": 256},
  {**BASE, "name": "depth4",         "num_hidden_layers": 4},
]

results = {}
models = {}

for cfg in configs:
  torch.manual_seed(RANDOM_SEED)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  model = FashionModel(
    input_features=NUM_FEATURES,
    hidden_units=cfg["hidden_units"],
    output_features=NUM_CLASSES,
    activation_fn=cfg["activation_fn"],
    num_hidden_layers=cfg["num_hidden_layers"],
    dropout=cfg["dropout"]
  ).to(device)
  apply_init(model, cfg["init"])

  optimizer = cfg["optimizer"](params=model.parameters(), **cfg["optimizer_kwargs"])

  print(f"\nSELECTED CONFIG: {cfg['name']}")
  results[cfg["name"]] = train_and_validate(model=model, optimizer=optimizer, epochs=30)
  models[cfg["name"]] = model

  h = results[cfg["name"]]
  print(f"{cfg['name']:<28}train acc {h['train_acc'][-1]:.2f}%   val acc {h['val_acc'][-1]:.2f}%")

import json
with open("results_mlp.json", "w") as f:
  json.dump(results, f)

ranking = sorted(results.items(), key=lambda item: item[1]["val_acc"][-1], reverse=True)


SELECTED CONFIG: relu_sgd_momentum
Epoch   Train Loss  Train Acc   Val Loss    Val Acc
0       0.73856     73.21       0.54648     80.93
1       0.45324     83.81       0.46535     82.72
2       0.40700     85.28       0.40030     85.55
3       0.37807     86.14       0.38019     86.13
4       0.35649     87.11       0.36149     86.77
5       0.33904     87.60       0.35998     87.03
6       0.32267     88.20       0.38371     86.27
7       0.31274     88.54       0.35904     87.00
8       0.30427     88.82       0.34628     87.28
9       0.29290     89.14       0.32344     88.17
10      0.28385     89.65       0.33581     87.83
11      0.27914     89.71       0.37570     86.83
12      0.27182     89.97       0.32376     87.98
13      0.26475     90.16       0.33120     88.28
14      0.26139     90.38       0.32457     88.77
15      0.25208     90.64       0.32061     88.68
16      0.24874     90.80       0.32236     88.68
17      0.24278     91.02       0.33165     88.25
18      0.23

In [86]:
print("\nLeaderboard (Val Acc%)")
for name, history in ranking:
  print(f"{name:<24}{history['val_acc'][-1]:.2f}%")


Leaderboard (Val Acc%)
hidden256               89.25%
gelu_adamw              89.00%
leaky_relu_adamw        88.90%
leaky_relu_sgd_momentum 88.88%
depth4                  88.75%
init_normal             88.65%
relu_sgd_momentum       88.57%
gelu_adam               88.52%
dropout0.3              88.48%
leaky_relu_adam         88.45%
wd0.1                   88.35%
init_kaiming            88.30%
relu_adamw              88.17%
lr3e-3                  88.03%
relu_adam               87.93%
gelu_sgd_momentum       87.93%


In [87]:
print("\nFinalists on Test Set")
test_results = {}
for name, history in ranking[:3]:
  test_loss, test_acc = test(models[name])
  test_results[name] = {"val_acc": history["val_acc"][-1], "test_acc": test_acc, "test_loss": test_loss}
  print(f"{name:<24}val acc {history['val_acc'][-1]:.2f}%   test acc {test_acc:.2f}%   test loss {test_loss:.5f}")
with open("test_mlp.json", "w") as f:
  json.dump(test_results, f)


Finalists on Test Set
hidden256               val acc 89.25%   test acc 89.08%   test loss 0.49413
gelu_adamw              val acc 89.00%   test acc 88.48%   test loss 0.38166
leaky_relu_adamw        val acc 88.90%   test acc 88.25%   test loss 0.36956


# Part (b)

## CNN model

### Architecture

In [88]:
class CNNModel(nn.Module):
  def __init__(self, in_channels, image_size, output_features,
               num_conv_blocks=2, filters=16, kernel_size=3,
               hidden_units=64, activation_fn=nn.ReLU, dropout=0.0):
    super().__init__()

    blocks = []
    channels = in_channels
    for _ in range(num_conv_blocks):
      blocks.append(nn.Conv2d(channels, filters, kernel_size=kernel_size, padding=kernel_size // 2))
      blocks.append(activation_fn())
      blocks.append(nn.MaxPool2d(2))
      channels = filters
    self.features = nn.Sequential(*blocks)

    with torch.no_grad():
      flat_size = self.features(torch.zeros(1, in_channels, image_size, image_size)).numel()

    head = [nn.Flatten(), nn.Linear(flat_size, hidden_units), activation_fn()]
    if dropout > 0.0:
      head.append(nn.Dropout(dropout))
    head.append(nn.Linear(hidden_units, output_features))
    self.classifier = nn.Sequential(*head)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.classifier(self.features(x))

## Global Variables

In [89]:
IN_CHANNELS = 1
IMAGE_SIZE  = 28

## Init

In [90]:
def apply_init_cnn(model: nn.Module, method: str):
  if method == "default":
    return

  for layer in model.modules():
    if isinstance(layer, (nn.Conv2d, nn.Linear)):
      if method == "kaiming_normal":
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
      elif method == "normal_0.01":
        nn.init.normal_(layer.weight, mean=0.0, std=0.01)
      elif method == "zeros":
        nn.init.zeros_(layer.weight)
      nn.init.zeros_(layer.bias)

## Run Script

In [91]:
activation_functions = {
  "relu":         nn.ReLU,
  "leaky_relu":   nn.LeakyReLU,
  "gelu":         nn.GELU
}

optimizers_and_lrs = [
  {"name": "sgd_momentum", "fn": torch.optim.SGD,   "kwargs": {"lr": 0.01, "momentum": 0.9}},
  {"name": "adam",         "fn": torch.optim.Adam,  "kwargs": {"lr": 0.001}},
  {"name": "adamw",        "fn": torch.optim.AdamW, "kwargs": {"lr": 0.001}},
]

BASE = {
  "activation_fn":     nn.GELU,
  "optimizer":         torch.optim.AdamW,
  "optimizer_kwargs":  {"lr": 1e-3, "weight_decay": 0.01},
  "num_conv_blocks":   2,
  "filters":           16,
  "kernel_size":       3,
  "hidden_units":      64,
  "dropout":           0.0,
  "init":              "default",
}

configs = []

for activation_name, activation_fn in activation_functions.items():
  for opt in optimizers_and_lrs:
    configs.append({
      **BASE,
      "name":             f"cnn_{activation_name}_{opt['name']}",
      "activation_fn":    activation_fn,
      "optimizer":        opt["fn"],
      "optimizer_kwargs": opt["kwargs"],
    })

configs += [
  {**BASE, "name": "cnn_lr3e-3",       "optimizer_kwargs": {"lr": 3e-3, "weight_decay": 0.01}},
  {**BASE, "name": "cnn_lr3e-4",       "optimizer_kwargs": {"lr": 3e-4, "weight_decay": 0.01}},
  {**BASE, "name": "cnn_wd0.0",        "optimizer_kwargs": {"lr": 1e-3, "weight_decay": 0.0}},
  {**BASE, "name": "cnn_wd0.1",        "optimizer_kwargs": {"lr": 1e-3, "weight_decay": 0.1}},
  {**BASE, "name": "cnn_dropout0.3",   "dropout": 0.3},
  {**BASE, "name": "cnn_init_normal",  "init": "normal_0.01"},
  {**BASE, "name": "cnn_init_kaiming", "init": "kaiming_normal"},
  {**BASE, "name": "cnn_filters8",     "filters": 8},
  {**BASE, "name": "cnn_filters32",    "filters": 32},
  {**BASE, "name": "cnn_kernel5",      "kernel_size": 5},
  {**BASE, "name": "cnn_blocks1",      "num_conv_blocks": 1},
  {**BASE, "name": "cnn_blocks3",      "num_conv_blocks": 3},
]

results = {}
models = {}

for cfg in configs:
  torch.manual_seed(RANDOM_SEED)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  model = CNNModel(
    in_channels=IN_CHANNELS,
    image_size=IMAGE_SIZE,
    output_features=NUM_CLASSES,
    num_conv_blocks=cfg["num_conv_blocks"],
    filters=cfg["filters"],
    kernel_size=cfg["kernel_size"],
    hidden_units=cfg["hidden_units"],
    activation_fn=cfg["activation_fn"],
    dropout=cfg["dropout"]
  ).to(device)
  apply_init_cnn(model, cfg["init"])

  optimizer = cfg["optimizer"](params=model.parameters(), **cfg["optimizer_kwargs"])

  print(f"\nSELECTED CONFIG: {cfg['name']}")
  results[cfg["name"]] = train_and_validate(model=model, optimizer=optimizer, epochs=30)
  models[cfg["name"]] = model

  h = results[cfg["name"]]
  print(f"{cfg['name']:<28}train acc {h['train_acc'][-1]:.2f}%   val acc {h['val_acc'][-1]:.2f}%")

import json
with open("results_cnn.json", "w") as f:
  json.dump(results, f)

ranking = sorted(results.items(), key=lambda item: item[1]["val_acc"][-1], reverse=True)


SELECTED CONFIG: cnn_relu_sgd_momentum
Epoch   Train Loss  Train Acc   Val Loss    Val Acc
0       0.72027     73.14       0.47544     82.82
1       0.41606     84.84       0.39926     85.68
2       0.36120     87.02       0.34258     87.42
3       0.32820     88.11       0.32210     87.88
4       0.30494     88.83       0.30395     88.78
5       0.28395     89.60       0.28781     89.22
6       0.26822     90.22       0.27692     89.93
7       0.25614     90.55       0.26193     90.53
8       0.24432     91.08       0.27471     90.18
9       0.23501     91.47       0.27406     89.42
10      0.22687     91.61       0.24906     90.57
11      0.21993     91.83       0.25068     91.00
12      0.21073     92.18       0.25757     90.70
13      0.20271     92.41       0.25276     90.60
14      0.19497     92.69       0.24360     91.08
15      0.19117     93.00       0.28239     89.55
16      0.18618     93.04       0.23942     91.45
17      0.18057     93.42       0.25609     91.12
18      

In [92]:
print("\nLeaderboard (Val Acc%)")
for name, history in ranking:
  print(f"{name:<28}{history['val_acc'][-1]:.2f}%")


Leaderboard (Val Acc%)
cnn_dropout0.3              91.98%
cnn_leaky_relu_adam         91.72%
cnn_relu_adamw              91.70%
cnn_relu_adam               91.38%
cnn_leaky_relu_adamw        91.37%
cnn_kernel5                 91.27%
cnn_gelu_adam               91.25%
cnn_wd0.0                   91.25%
cnn_filters32               91.25%
cnn_gelu_adamw              91.07%
cnn_init_normal             91.00%
cnn_lr3e-3                  90.88%
cnn_blocks1                 90.87%
cnn_lr3e-4                  90.85%
cnn_wd0.1                   90.77%
cnn_relu_sgd_momentum       90.70%
cnn_blocks3                 90.58%
cnn_filters8                90.55%
cnn_init_kaiming            90.37%
cnn_gelu_sgd_momentum       90.13%
cnn_leaky_relu_sgd_momentum 89.92%


In [93]:
print("\nFinalists on Test Set")
test_results = {}
for name, history in ranking[:3]:
  test_loss, test_acc = test(models[name])
  test_results[name] = {"val_acc": history["val_acc"][-1], "test_acc": test_acc, "test_loss": test_loss}
  print(f"{name:<24}val acc {history['val_acc'][-1]:.2f}%   test acc {test_acc:.2f}%   test loss {test_loss:.5f}")
with open("test_cnn.json", "w") as f:
  json.dump(test_results, f)


Finalists on Test Set
cnn_dropout0.3          val acc 91.98%   test acc 91.52%   test loss 0.27634
cnn_leaky_relu_adam     val acc 91.72%   test acc 91.54%   test loss 0.33405
cnn_relu_adamw          val acc 91.70%   test acc 91.28%   test loss 0.29416


# Applied to CIFAR10

In [94]:
from torchvision.transforms import v2

## Global Variables

In [95]:
EPOCHS = 30

IN_CHANNELS  = 3
IMAGE_SIZE   = 32
NUM_FEATURES = IN_CHANNELS * IMAGE_SIZE * IMAGE_SIZE   # 3072, for the MLP

## Configurations

In [96]:
CONFIGS = [
  {
    "name": "hidden256", "network": "mlp",
    "activation_fn": nn.GELU, "optimizer": torch.optim.AdamW,
    "optimizer_kwargs": {"lr": 1e-3, "weight_decay": 0.01},
    "hidden_units": 256, "num_hidden_layers": 2, "dropout": 0.0, "init": "default",
  },
  {
    "name": "cnn_dropout0.3", "network": "cnn",
    "activation_fn": nn.GELU, "optimizer": torch.optim.AdamW,
    "optimizer_kwargs": {"lr": 1e-3, "weight_decay": 0.01},
    "num_conv_blocks": 2, "filters": 16, "kernel_size": 3, "hidden_units": 64, "dropout": 0.3, "init": "default",
  },
  {
    "name": "cnn_leaky_relu_adam", "network": "cnn",
    "activation_fn": nn.LeakyReLU, "optimizer": torch.optim.Adam,
    "optimizer_kwargs": {"lr": 1e-3},
    "num_conv_blocks": 2, "filters": 16, "kernel_size": 3, "hidden_units": 64, "dropout": 0.0, "init": "default",
  },
]

## Run Script

In [97]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

full_train = CIFAR10(root="data", train=True,  download=True, transform=transform)
test_data  = CIFAR10(root="data", train=False, download=True, transform=transform)

# Same 90/10 split as Fashion-MNIST: 45,000 train / 5,000 validation
train_data, val_data = random_split(
  full_train,
  [0.9, 0.1],
  generator=torch.Generator().manual_seed(RANDOM_SEED)
)
print(f"Train: {len(train_data)}  Validation: {len(val_data)}  Test: {len(test_data)}")

train_data, val_data, test_data = train_data, val_data, test_data
train_dataloader = move_to_gpu(train_data, device, shuffle=True)
val_dataloader   = move_to_gpu(val_data, device, shuffle=False)
test_dataloader  = move_to_gpu(test_data, device, shuffle=False)

results = {}
models = {}

for cfg in CONFIGS:
  torch.manual_seed(RANDOM_SEED)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  if cfg["network"] == "mlp":
    model = FashionModel(
      input_features=NUM_FEATURES,
      hidden_units=cfg["hidden_units"],
      output_features=NUM_CLASSES,
      activation_fn=cfg["activation_fn"],
      num_hidden_layers=cfg["num_hidden_layers"],
      dropout=cfg["dropout"]
    ).to(device)
    apply_init(model, cfg["init"])
  else:
    model = CNNModel(
      in_channels=IN_CHANNELS,
      image_size=IMAGE_SIZE,
      output_features=NUM_CLASSES,
      num_conv_blocks=cfg["num_conv_blocks"],
      filters=cfg["filters"],
      kernel_size=cfg["kernel_size"],
      hidden_units=cfg["hidden_units"],
      activation_fn=cfg["activation_fn"],
      dropout=cfg["dropout"]
    ).to(device)
    apply_init_cnn(model, cfg["init"])

  optimizer = cfg["optimizer"](params=model.parameters(), **cfg["optimizer_kwargs"])

  print(f"\nSELECTED CONFIG: {cfg['name']}")
  results[cfg["name"]] = train_and_validate(model=model, optimizer=optimizer, epochs=EPOCHS)
  models[cfg["name"]] = model

  h = results[cfg["name"]]
  print(f"{cfg['name']:<28}train acc {h['train_acc'][-1]:.2f}%   val acc {h['val_acc'][-1]:.2f}%")

with open("results_cifar.json", "w") as f:
  json.dump(results, f)

Files already downloaded and verified
Files already downloaded and verified
Train: 45000  Validation: 5000  Test: 10000

SELECTED CONFIG: hidden256
Epoch   Train Loss  Train Acc   Val Loss    Val Acc
0       1.83901     33.20       1.69629     39.50
1       1.65060     40.88       1.64372     41.52
2       1.56193     43.91       1.54380     44.68
3       1.50080     46.42       1.53598     45.24
4       1.45831     47.74       1.52532     45.86
5       1.41967     49.46       1.48576     47.74
6       1.39140     49.93       1.49209     47.64
7       1.36450     51.32       1.41010     49.62
8       1.34984     51.57       1.45831     48.86
9       1.31692     52.82       1.47537     47.74
10      1.30117     53.45       1.45549     48.82
11      1.27622     54.38       1.46531     49.08
12      1.26518     54.65       1.39641     50.64
13      1.24344     55.32       1.44524     49.24
14      1.22822     55.88       1.44682     49.94
15      1.20622     56.65       1.41675     50.42


In [98]:
print("\nCIFAR-10 test set")
test_results = {}
for name, history in results.items():
  test_loss, test_acc = test(models[name])
  test_results[name] = {"val_acc": history["val_acc"][-1], "test_acc": test_acc, "test_loss": test_loss}
  print(f"{name:<28}val acc {history['val_acc'][-1]:.2f}%   test acc {test_acc:.2f}%   test loss {test_loss:.5f}")

with open("test_cifar.json", "w") as f:
  json.dump(test_results, f)


CIFAR-10 test set
hidden256                   val acc 49.86%   test acc 50.76%   test loss 1.47164
cnn_dropout0.3              val acc 66.94%   test acc 67.77%   test loss 1.00614
cnn_leaky_relu_adam         val acc 66.50%   test acc 65.43%   test loss 1.28980


# Plotting Script

## Imports

In [99]:
import os

## Global Variables

In [100]:
OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

## Helper Functions

In [101]:
def load(path):
  if os.path.exists(path):
    with open(path) as f:
      return json.load(f)
  return None


def save(fig, fname):
  path = os.path.join(OUT_DIR, fname)
  fig.savefig(path, dpi=150, bbox_inches="tight")
  plt.close(fig)
  plt.show(fig)
  print(f"saved {path}")


def val_acc(res, name):
  return res[name]["val_acc"][-1]

def print_tables(nets):
  for label, res in nets.items():
    print(f"\n{label}: top 5 by final validation accuracy")
    for n in sorted(res, key=lambda n: val_acc(res, n), reverse=True)[:5]:
      print(f"  {n:<26}train {res[n]['train_acc'][-1]:.2f}%   val {val_acc(res, n):.2f}%")

  for label, path in [("MLP", "test_mlp.json"), ("CNN", "test_cnn.json"), ("CIFAR-10", "test_cifar.json")]:
    test = load(path)
    if test:
      print(f"\n{label}: finalists on the test set")
      for n, r in test.items():
        print(f"  {n:<26}val {r['val_acc']:.2f}%   test {r['test_acc']:.2f}%   loss {r['test_loss']:.4f}")

## Learning Curves

In [102]:
def plot_curves(nets):
  fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(13, 5))
  for k, (label, res) in enumerate(nets.items()):
    best = max(res, key=lambda n: val_acc(res, n))
    h = res[best]
    epochs = range(1, len(h["val_acc"]) + 1)
    ax_loss.plot(epochs, h["val_loss"], color=f"C{k}", label=f"{label} val ({best})")
    ax_loss.plot(epochs, h["train_loss"], color=f"C{k}", linestyle="--", alpha=0.6, label=f"{label} train")
    ax_acc.plot(epochs, h["val_acc"], color=f"C{k}", label=f"{label} val ({best})")
    ax_acc.plot(epochs, h["train_acc"], color=f"C{k}", linestyle="--", alpha=0.6, label=f"{label} train")
  ax_loss.set_title("Loss per epoch (best config per network)")
  ax_acc.set_title("Accuracy (%) per epoch (best config per network)")
  for ax in (ax_loss, ax_acc):
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
  save(fig, "curves.png")

## Final Validation Accuracy

In [103]:
def plot_configs(nets):
  fig, axes = plt.subplots(1, len(nets), figsize=(7 * len(nets), 7), squeeze=False)
  for ax, (label, res) in zip(axes[0], nets.items()):
    names = sorted(res, key=lambda n: val_acc(res, n))
    accs = [val_acc(res, n) for n in names]
    ax.barh(names, accs)
    ax.set_xlim(min(accs) - 1, max(accs) + 0.5)
    ax.set_xlabel("final validation accuracy (%)")
    ax.set_title(label)
    ax.grid(alpha=0.3, axis="x")
  save(fig, "configs.png")

## Carry-over / Transfer Experiment

In [104]:
def plot_transfer(fashion, cifar):
  names = [n for n in cifar if n in fashion]
  if not names:
    print("No config names shared between the CIFAR-10 and Fashion-MNIST results; skipping transfer.png")
    return
  fig, ax = plt.subplots(figsize=(1.8 * len(names) + 3, 5))
  xs = range(len(names))
  ax.bar([x - 0.2 for x in xs], [val_acc(fashion, n) for n in names], width=0.4, label="Fashion-MNIST")
  ax.bar([x + 0.2 for x in xs], [val_acc(cifar, n) for n in names], width=0.4, label="CIFAR-10")
  ax.set_xticks(list(xs))
  ax.set_xticklabels(names, rotation=20, ha="right")
  ax.set_ylabel("final validation accuracy (%)")
  ax.legend()
  ax.grid(alpha=0.3, axis="y")
  save(fig, "transfer.png")

## Main Script

In [105]:
mlp, cnn, cifar = load("results_mlp.json"), load("results_cnn.json"), load("results_cifar.json")
nets = {label: res for label, res in [("MLP", mlp), ("CNN", cnn)] if res}
if not nets:
	raise SystemExit("No results_mlp.json / results_cnn.json found. Save them from the experiment scripts first.")

plot_curves(nets)
plot_configs(nets)

if cifar:
	plot_transfer({**(mlp or {}), **(cnn or {})}, cifar)
else:
	print("results_cifar.json not found; skipping transfer.png")
print_tables(nets)

saved figures/curves.png
saved figures/configs.png
saved figures/transfer.png

MLP: top 5 by final validation accuracy
  hidden256                 train 96.69%   val 89.25%
  gelu_adamw                train 94.10%   val 89.00%
  leaky_relu_adamw          train 93.72%   val 88.90%
  leaky_relu_sgd_momentum   train 92.51%   val 88.88%
  depth4                    train 94.37%   val 88.75%

CNN: top 5 by final validation accuracy
  cnn_dropout0.3            train 95.29%   val 91.98%
  cnn_leaky_relu_adam       train 97.12%   val 91.72%
  cnn_relu_adamw            train 96.02%   val 91.70%
  cnn_relu_adam             train 96.36%   val 91.38%
  cnn_leaky_relu_adamw      train 97.16%   val 91.37%

MLP: finalists on the test set
  hidden256                 val 89.25%   test 89.08%   loss 0.4941
  gelu_adamw                val 89.00%   test 88.48%   loss 0.3817
  leaky_relu_adamw          val 88.90%   test 88.25%   loss 0.3696

CNN: finalists on the test set
  cnn_dropout0.3            val 91.